# BEM Python API

The NGSolve BEM interface follows the same source, kernel, and test structure as the Galerkin formulation. The Helmholtz example below first maps the paper notation to code and then shows the main API controls.

## Implemented Capabilities

- Laplace, Helmholtz, Lame and Maxwell operators,
- real or complex densities, scalar or vector valued, and real or complex wavenumbers,
- triangular, quadrilateral and curved surface elements,
- direct or FMM evaluation on full or selected boundaries.

## Helmholtz Single Layer Potential

For an existing surface mesh, the discrete space and symbolic trial and test functions are

```python
from ngsolve import *
from ngsolve.bem import *

fes = SurfaceL2(mesh, order=order, complex=True)
rho, eta = fes.TnT()
kappa = 4.0
```

The single layer potential maps the density $\rho_h$ to

$$
(\widetilde V\rho_h)(x)
=\int_\Gamma G_\kappa(x,y)\rho_h(y)\,d\sigma_y.
$$

This source integral is constructed by

```python
SL = HelmholtzSL(rho * ds, kappa)
```

## From Potential to Galerkin Operator

Testing the potential on the boundary gives the variational problem: find $\rho_h \in X_h$ such that, for all $\eta_h \in X_h$,

$$
\langle V\rho_h,\eta_h\rangle_\Gamma
=\int_\Gamma \eta_h(x)(\widetilde V\rho_h)(x)\,d\sigma_x
=\int_\Gamma\int_\Gamma
\eta_h(x)G_\kappa(x,y)\rho_h(y)
\,d\sigma_y\,d\sigma_x
=\langle g,\eta_h\rangle_\Gamma.
$$

Multiplication by the test function and boundary measure performs the same step in code:

```python
V = SL * eta * ds
A = V.mat
```

| Paper | NGSolve |
| --- | --- |
| $X_h$ | `SurfaceL2(...)` |
| $\rho_h,\eta_h$ | `rho, eta = fes.TnT()` |
| $G_\kappa$ and the source integral | `HelmholtzSL(rho * ds, kappa)` |
| testing with $\eta_h$ | `* eta * ds` |
| Galerkin matrix action $A$ | `V.mat` |

Using `use_fmm=False` in `HelmholtzSL` switches to direct operator assembly for comparison.

## Boundaries and FMM Controls

The first boundary region contains the sources and the second contains the test functions:

```python
SL_region = HelmholtzSL(
    rho * ds("source"),
    kappa,
    use_fmm=True,
    fmm_minorder=20,
    fmm_separation=2.0,
)
V_region = SL_region * eta * ds("target")
```

The names `source` and `target` refer to named mesh boundaries. Omitting the name uses the full boundary.

The resulting operator provides

- `V_region.mat` for the matrix action,
- `V_region.GetFMMInfo()` for tree and expansion statistics,
- `V_region.NearFieldMatrix()` for accurately assembled touching element entries,
- `V_region.CalcSubMatrix(rows, cols)` for selected matrix entries.

`NearFieldMatrix()` is sparse because it contains only same panel, common edge, and common vertex interactions. In the FMM matrix action, these accurate entries replace the ordinary quadrature values for the same pairs.

## Right Hand Side and Solve

The right hand side $\langle g,\eta_h\rangle_\Gamma$ and the discrete density are

```python
rhs = LinearForm(g * eta * ds).Assemble()
rho_h = GridFunction(fes)

with TaskManager():
    rho_h.vec.data = solvers.GMRes(
        A=A, b=rhs.vec, tol=1e-8, maxsteps=200
    )
```

## Evaluate the Potential

The same potential object is applied to the computed density:

```python
field = SL(rho_h)
field_on_screen = SL(rho_h, target_boundary)
grad_field = grad(SL)(rho_h)
```

The second form builds a local expansion for the target boundary region. Differentiated potentials are available when the kernel provides the corresponding operation.

## Vector Valued Operators

Vector valued kernels use the same construction pattern. For the Lame single layer operator:

```python
fes_vec = VectorH1(mesh, order=order)
t, s = fes_vec.TnT()

L = LameSL(t * ds, E=E, nu=nu) * s * ds
```

Maxwell operators follow the same pattern with `HDivSurface` or `HCurl` traces; the complete construction is shown in the Maxwell example.

## Other Kernels, Same Pattern

| Mathematical operator | NGSolve building block |
| --- | --- |
| Laplace single layer | `LaplaceSL` |
| Laplace double layer | `LaplaceDL` |
| Helmholtz single layer | `HelmholtzSL` |
| Helmholtz double layer | `HelmholtzDL` |
| Helmholtz combined field | `HelmholtzCF` |
| Elasticity single layer | `LameSL` |
| Maxwell double layer | `MaxwellDL` |

Vector valued formulations use the same source and test pattern with the appropriate trace operators.